In [1]:
import numpy as np

In [2]:
def dekomposisi_lu_doolittle(A, B):
    """
    Menyelesaikan AX = B dengan Faktorisasi LU (Metode Doolittle).
    L memiliki diagonal utama bernilai 1.
    """
    n = len(A)
    L = np.zeros((n, n))
    U = np.zeros((n, n))
    
    # 1. TAHAP DEKOMPOSISI (A = L * U)
    for i in range(n):
        L[i, i] = 1.0  # Aturan Doolittle
        
        # Mengisi Matriks Segitiga Atas (U)
        for k in range(i, n):
            s1 = sum(L[i, j] * U[j, k] for j in range(i))
            U[i, k] = A[i, k] - s1
            
        # Mengisi Matriks Segitiga Bawah (L)
        for k in range(i + 1, n):
            s2 = sum(L[k, j] * U[j, i] for j in range(i))
            L[k, i] = (A[k, i] - s2) / U[i, i]
            
    # 2. SUBSTITUSI MAJU (L * Y = B)
    Y = np.zeros(n)
    for i in range(n):
        s3 = sum(L[i, j] * Y[j] for j in range(i))
        Y[i] = B[i] - s3
        
    # 3. SUBSTITUSI MUNDUR (U * X = Y)
    X = np.zeros(n)
    for i in range(n - 1, -1, -1):
        s4 = sum(U[i, j] * X[j] for j in range(i + 1, n))
        X[i] = (Y[i] - s4) / U[i, i]
        
    return L, U, X

In [3]:
def dekomposisi_cholesky(A, B):
    """
    Menyelesaikan AX = B dengan Faktorisasi Cholesky (A = L * L^T).
    Syarat: A harus Simetris dan Definit Positif.
    """
    n = len(A)
    L = np.zeros((n, n))
    
    # 1. TAHAP DEKOMPOSISI
    for i in range(n):
        for j in range(i + 1):
            s = sum(L[i, k] * L[j, k] for k in range(j))
            
            if i == j:  # Elemen Diagonal Utama
                L[i, j] = np.sqrt(A[i, i] - s)
            else:      # Elemen di Luar Diagonal Utama
                L[i, j] = (A[i, j] - s) / L[j, j]
                
    # 2. SUBSTITUSI MAJU (L * Y = B)
    Y = np.zeros(n)
    for i in range(n):
        s3 = sum(L[i, j] * Y[j] for j in range(i))
        Y[i] = (B[i] - s3) / L[i, i]
        
    # 3. SUBSTITUSI MUNDUR (L^T * X = Y)
    LT = L.T
    X = np.zeros(n)
    for i in range(n - 1, -1, -1):
        s4 = sum(LT[i, j] * X[j] for j in range(i + 1, n))
        X[i] = (Y[i] - s4) / LT[i, i]
        
    return L, X

In [4]:
# =====================================================================
# DEMO STUDI KASUS TEKNIK MESIN
# =====================================================================
if __name__ == "__main__":
    print("-" * 60)
    print("KASUS 1: DEKOMPOSISI LU (Analisis Truss / Beban Variabel)")
    print("-" * 60)
    # Matriks Kekakuan Struktur Rangka (3x3)
    A_truss = np.array([[4.0, -1.0, -1.0],
                        [-1.0, 3.0, 0.0],
                        [-1.0, 0.0, 5.0]])
    
    # Skenario Beban Eksternal Sisi Atas dan Sisi Samping (kN)
    B_beban1 = np.array([12.0, 10.0, 15.0])
    B_beban2 = np.array([20.0, 5.0,  0.0]) # Cukup panggil ulang forward-backward substitusi
    
    L, U, X1 = dekomposisi_lu_doolittle(A_truss, B_beban1)
    print("Matriks L:\n", L)
    print("Matriks U:\n", U)
    print(f"Solusi Defleksi untuk Beban Skenario 1: {X1}\n")

    L, U, X2 = dekomposisi_lu_doolittle(A_truss, B_beban2)
    print("Matriks L:\n", L)
    print("Matriks U:\n", U)
    print(f"Solusi Defleksi untuk Beban Skenario 2: {X2}\n")

    print("-" * 60)
    print("KASUS 2: DEKOMPOSISI CHOLESKY (Matriks Simetris Definit Positif)")
    print("-" * 60)
    # Matriks Kekakuan Global (FEA) Solid Element - Selalu Simetris
    A_simetris = np.array([[6.0, 15.0, 55.0],
                           [15.0, 55.0, 225.0],
                           [55.0, 225.0, 979.0]])
    B_gaya = np.array([76.0, 295.0, 1259.0])
    
    L_cholesky, X_gaya = dekomposisi_cholesky(A_simetris, B_gaya)
    print("Matriks L (Cholesky):\n", L_cholesky)
    print(f"Solusi Distribusi Perpindahan Perpindahan Gaya: {X_gaya}")

------------------------------------------------------------
KASUS 1: DEKOMPOSISI LU (Analisis Truss / Beban Variabel)
------------------------------------------------------------
Matriks L:
 [[ 1.          0.          0.        ]
 [-0.25        1.          0.        ]
 [-0.25       -0.09090909  1.        ]]
Matriks U:
 [[ 4.         -1.         -1.        ]
 [ 0.          2.75       -0.25      ]
 [ 0.          0.          4.72727273]]
Solusi Defleksi untuk Beban Skenario 1: [5.28846154 5.09615385 4.05769231]

Matriks L:
 [[ 1.          0.          0.        ]
 [-0.25        1.          0.        ]
 [-0.25       -0.09090909  1.        ]]
Matriks U:
 [[ 4.         -1.         -1.        ]
 [ 0.          2.75       -0.25      ]
 [ 0.          0.          4.72727273]]
Solusi Defleksi untuk Beban Skenario 2: [6.25 3.75 1.25]

------------------------------------------------------------
KASUS 2: DEKOMPOSISI CHOLESKY (Matriks Simetris Definit Positif)
----------------------------------------